# 02 — Feature Engineering & Target Engineering (v2)

Esta versão **reconstrói completamente** o pipeline de features e o alvo (target) para o modelo de ML.

## O que mudou em relação à v1

| v1 (IRR) | v2 (esta versão) |
|---|---|
| Target = índice ponderado manualmente (IRR) | Target = proxy físico derivado dos dados |
| Pesos arbitrários (50/30/20%) | Sem pesos manuais |
| Risco por faixa agregada | Risco **por objeto** individual |
| Lógica circular (target = f(features)) | Sem vazamento de dados |

## Estrutura do notebook

1. Carregamento e limpeza dos dados
2. Features orbitais base
3. Features físicas derivadas
4. Estratégia de agregação (objeto e faixa)
5. **Target Engineering** — proxy de risco sem pesos arbitrários
6. Features avançadas (densidade local, dispersão)
7. Validação de features (correlação, distribuição)
8. Dataset final para ML
9. Documentação das escolhas

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.neighbors import BallTree
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 110

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')
OUT_DIR  = Path('../outputs')
PROC_DIR.mkdir(exist_ok=True)

print('Imports OK')

## 1. Carregamento e Limpeza

Partimos do `gp_leo.csv` — elementos orbitais de objetos em LEO baixados do Space-Track.

Limpeza necessária:
- Converter colunas numéricas que podem ter vindo como string
- Calcular altitude a partir do semi-eixo maior (SEMIMAJOR_AXIS) ou do apogeu/perigeu
- Remover objetos com dados críticos faltando

In [ ]:
gp_raw = pd.read_csv(RAW_DIR / 'gp_leo.csv', low_memory=False)
print(f'Raw: {gp_raw.shape[0]:,} linhas x {gp_raw.shape[1]} colunas')

# Colunas numéricas que precisamos
NUM_COLS = [
    'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION',
    'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS',
    'BSTAR', 'MEAN_MOTION_DOT', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER'
]
for col in NUM_COLS:
    if col in gp_raw.columns:
        gp_raw[col] = pd.to_numeric(gp_raw[col], errors='coerce')

# Verificar missing values nas colunas críticas
missing = gp_raw[NUM_COLS].isnull().sum()
print('\nMissing values nas colunas numericas:')
print(missing[missing > 0])

In [ ]:
# Manter apenas colunas necessárias
cols_manter = [
    'NORAD_CAT_ID', 'OBJECT_NAME', 'OBJECT_TYPE', 'COUNTRY_CODE',
    'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'SEMIMAJOR_AXIS',
    'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'MEAN_MOTION_DOT',
    'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'LAUNCH_DATE'
]
cols_manter = [c for c in cols_manter if c in gp_raw.columns]
gp = gp_raw[cols_manter].copy()

# Remover objetos sem dados orbitais críticos
gp = gp.dropna(subset=['MEAN_MOTION', 'INCLINATION', 'ECCENTRICITY'])
gp = gp[gp['MEAN_MOTION'] > 0]
gp = gp[gp['ECCENTRICITY'] >= 0]

print(f'Apos limpeza: {len(gp):,} objetos')
print(f'Removidos: {len(gp_raw) - len(gp):,} objetos com dados criticos faltando')

## 2. Features Orbitais Base

Convertemos os elementos TLE em features físicas interpretáveis.

### Altitude

O Space-Track já fornece `APOAPSIS` e `PERIAPSIS` em km. Usamos a média como altitude representativa.
Se não disponível, calculamos a partir do semi-eixo maior: `altitude = SEMIMAJOR_AXIS - R_TERRA`.

### Período orbital

Já fornecido em `PERIOD` (minutos). Se não, calculamos de `MEAN_MOTION` (revoluções/dia):
`period_min = 1440 / mean_motion`

In [ ]:
R_TERRA = 6371.0  # km

# Altitude média (km)
if 'APOAPSIS' in gp.columns and 'PERIAPSIS' in gp.columns:
    gp['altitude_km'] = (gp['APOAPSIS'] + gp['PERIAPSIS']) / 2
else:
    gp['altitude_km'] = gp['SEMIMAJOR_AXIS'] - R_TERRA

# Garantir que estamos em LEO (200-2000 km)
gp = gp[(gp['altitude_km'] >= 200) & (gp['altitude_km'] <= 2000)].copy()

# Período orbital (minutos)
if 'PERIOD' in gp.columns:
    gp['period_min'] = gp['PERIOD']
else:
    gp['period_min'] = 1440.0 / gp['MEAN_MOTION']

# Distância radial ao centro da Terra (km)
gp['radial_dist_km'] = R_TERRA + gp['altitude_km']

# Inclinação e excentricidade já estão em graus e adimensional
gp['inclination_deg'] = gp['INCLINATION']
gp['eccentricity']    = gp['ECCENTRICITY']

print(f'Objetos em LEO validos: {len(gp):,}')
print(f'Altitude: {gp["altitude_km"].min():.0f} - {gp["altitude_km"].max():.0f} km')
print(f'Periodo: {gp["period_min"].min():.1f} - {gp["period_min"].max():.1f} min')
print(f'Inclinacao: {gp["inclination_deg"].min():.1f} - {gp["inclination_deg"].max():.1f} deg')

## 3. Features Físicas Derivadas

### Velocidade orbital aproximada

Usamos a velocidade circular aproximada:

$$v = \sqrt{\frac{\mu}{r}}$$

onde $\mu = 398600.4$ km³/s² (parâmetro gravitacional da Terra) e $r$ = distância radial.

Esta é a velocidade típica de um objeto naquela altitude. Em colisões, o que importa é a **velocidade relativa** entre dois objetos — que depende das inclinações. Esse cálculo vem depois.

### Drag ballistic (BSTAR)

BSTAR é um coeficiente de drag da TLE. Objetos com BSTAR alto perdem altitude mais rápido — são mais instáveis e difíceis de prever. É um proxy indireto de área/massa (objetos grandes ou leves têm drag maior).

In [ ]:
MU_TERRA = 398600.4  # km^3/s^2

# Velocidade orbital circular aproximada (km/s)
gp['velocity_km_s'] = np.sqrt(MU_TERRA / gp['radial_dist_km'])

# BSTAR: coeficiente de drag (pode ser negativo ou zero — tratar)
if 'BSTAR' in gp.columns:
    gp['bstar'] = gp['BSTAR'].fillna(0.0)
    gp['bstar_abs'] = np.abs(gp['bstar'])  # magnitude do drag
else:
    gp['bstar'] = 0.0
    gp['bstar_abs'] = 0.0

# Taxa de decaimento orbital (rad/s² proxy)
if 'MEAN_MOTION_DOT' in gp.columns:
    gp['decay_rate'] = gp['MEAN_MOTION_DOT'].fillna(0.0)
else:
    gp['decay_rate'] = 0.0

print('Features fisicas criadas:')
print(gp[['altitude_km','velocity_km_s','period_min','inclination_deg','eccentricity','bstar_abs']].describe().round(3))

## 4. Features Categóricas

### Tipo de objeto

Esta é a feature categórica mais importante. Debris é incontrolável e mais perigoso.
Codificamos como variáveis numéricas ordinais baseadas em controlabilidade:

| Tipo | Código | Justificativa |
|---|---|---|
| PAYLOAD | 0 | Controlável (pode fazer manobras) |
| ROCKET BODY | 1 | Incontrolável mas grande e rastreável |
| DEBRIS | 2 | Incontrolável, pequeno, difícil de rastrear |
| UNKNOWN/TBA | 3 | Mais incerto — máximo risco por incerteza |

In [ ]:
# Normalizar nomes dos tipos
gp['OBJECT_TYPE'] = gp['OBJECT_TYPE'].fillna('UNKNOWN').str.upper().str.strip()

# Mapeamento ordinal de controlabilidade
tipo_map = {
    'PAYLOAD':     0,
    'ROCKET BODY': 1,
    'DEBRIS':      2,
    'UNKNOWN':     3,
    'TBA':         3,
}
gp['object_type_code'] = gp['OBJECT_TYPE'].map(tipo_map).fillna(3).astype(int)

# Flag binária: é debris ou não?
gp['is_debris'] = (gp['OBJECT_TYPE'] == 'DEBRIS').astype(int)
gp['is_uncontrolled'] = (gp['object_type_code'] >= 1).astype(int)

print('Distribuicao de tipos:')
print(gp['OBJECT_TYPE'].value_counts())
print(f'\nFracao incontrolavel: {gp["is_uncontrolled"].mean():.1%}')

## 5. Estratégia de Agregação

Trabalhamos em **dois níveis** conforme especificado:

### Nível 1 — Por objeto (preferido para ML)
Cada linha = um objeto. O target será calculado para cada objeto individualmente.

### Nível 2 — Por faixa de 50 km (secundário, para comparação)
Bins menores que os 100 km da v1 — captura gradientes mais finos de densidade.

In [ ]:
# Faixas de 50 km
bins_50 = list(range(200, 2050, 50))
labels_50 = [f'{b}-{b+50}' for b in bins_50[:-1]]

gp['faixa_50km'] = pd.cut(gp['altitude_km'], bins=bins_50,
                           labels=labels_50, right=False)
gp['faixa_50km_str'] = gp['faixa_50km'].astype(str)

# Centro da faixa como numérico
gp['faixa_centro'] = pd.cut(gp['altitude_km'], bins=bins_50,
                              labels=[b+25 for b in bins_50[:-1]],
                              right=False).astype(float)

print(f'Bins de 50 km criados: {len(labels_50)}')
print(f'Distribuicao nas top 10 faixas:')
print(gp['faixa_50km_str'].value_counts().head(10))

## 6. Target Engineering — Proxy de Risco de Colisão

Esta é a parte mais crítica. Diferentemente da v1 (IRR com pesos arbitrários), aqui o target é **derivado dos dados com justificativa física**.

### Por que o IRR era problemático?

O IRR era uma soma ponderada onde os pesos (50/30/20%) foram definidos manualmente. Isso cria **circularidade**: se usarmos `densidade` como feature e o target for `f(densidade)` com peso 50%, o modelo aprende trivialmente. Além disso, os pesos não têm fundamentação empírica.

### Nossa abordagem: Conjunction Proxy Score (CPS)

Baseado na formulação de taxa de colisão de Kessler, a frequência de colisões é proporcional a:

$$f_{colisao} \propto n_1 \cdot n_2 \cdot \sigma \cdot v_{rel}$$

onde:
- $n$ = densidade numérica de objetos (obj/km³)
- $\sigma$ = seção transversal de colisão (ignoramos — assumimos uniforme)
- $v_{rel}$ = velocidade relativa entre objetos

Para um objeto individual, o **risco de colisão local** é proporcional à **densidade de objetos vizinhos** × **velocidade relativa típica na vizinhança**.

A velocidade relativa entre dois objetos em órbitas com inclinações $i_1$ e $i_2$ é aproximada por:

$$v_{rel} \approx v_{orbital} \cdot \sqrt{2 - 2\cos(\Delta i)}$$

onde $\Delta i$ é a diferença de inclinação.

### Implementação: vizinhança espacial por altitude + inclinação

In [ ]:
# ── Passo 6.1: Densidade local por objeto ──────────────────────
#
# Para cada objeto, contamos quantos outros objetos estão a:
#   - ±50 km de altitude
#   - Qualquer inclinação (densidade bruta)
#
# Usamos lookup vetorizado em pandas (mais robusto que BallTree para 1D)

print('Calculando densidade local (vizinhos em +-50 km de altitude)...')
print('Isso pode levar 30-60 segundos para 27.000+ objetos...')

alt = gp['altitude_km'].values
N = len(alt)

# Vetorizado: para cada objeto, conta vizinhos em [alt-50, alt+50]
# Usamos numpy broadcasting
RAIO_ALT = 50.0  # km

# Para evitar uso excessivo de memória, fazemos em chunks
chunk = 1000
local_density = np.zeros(N, dtype=np.float32)

for start in range(0, N, chunk):
    end = min(start + chunk, N)
    alt_chunk = alt[start:end]  # shape (chunk,)
    # distância de cada objeto do chunk para todos os outros
    diffs = np.abs(alt_chunk[:, None] - alt[None, :])  # (chunk, N)
    # conta vizinhos (exclui o próprio objeto)
    counts = (diffs <= RAIO_ALT).sum(axis=1) - 1
    local_density[start:end] = counts

gp['local_density_raw'] = local_density

# Normalizar pela volume da casca de ±50 km
# V = (4/3) * pi * [(r + 50)^3 - (r - 50)^3]
r = gp['radial_dist_km'].values
vol = (4/3) * np.pi * ((r + RAIO_ALT)**3 - (r - RAIO_ALT)**3)
gp['local_density_km3'] = gp['local_density_raw'] / vol

print(f'Densidade local calculada para {N:,} objetos.')
print(f'  Minimo: {gp["local_density_km3"].min():.2e} obj/km3')
print(f'  Maximo: {gp["local_density_km3"].max():.2e} obj/km3')
print(f'  Mediana: {gp["local_density_km3"].median():.2e} obj/km3')

In [ ]:
# ── Passo 6.2: Densidade local de DEBRIS ──────────────────────
#
# Para cada objeto, conta apenas DEBRIS vizinhos (mais perigosos)
# Mesma janela de ±50 km de altitude

print('Calculando densidade local de debris...')

alt_debris = gp.loc[gp['is_debris'] == 1, 'altitude_km'].values
alt_all    = gp['altitude_km'].values
N = len(alt_all)

debris_density = np.zeros(N, dtype=np.float32)

for start in range(0, N, chunk):
    end = min(start + chunk, N)
    alt_chunk = alt_all[start:end]
    diffs = np.abs(alt_chunk[:, None] - alt_debris[None, :])
    counts = (diffs <= RAIO_ALT).sum(axis=1)
    debris_density[start:end] = counts

gp['debris_density_raw'] = debris_density
gp['debris_density_km3'] = gp['debris_density_raw'] / vol

# Fração de debris na vizinhança
gp['debris_fraction_local'] = np.where(
    gp['local_density_raw'] > 0,
    gp['debris_density_raw'] / gp['local_density_raw'],
    0.0
)

print(f'Debris density calculada.')
print(f'  Fracao de debris local (media): {gp["debris_fraction_local"].mean():.1%}')

In [ ]:
# ── Passo 6.3: Velocidade relativa proxy ────────────────────────
#
# v_rel entre dois objetos com inclinações i1 e i2:
#   v_rel ≈ v_orbital * sqrt(2 - 2*cos(delta_i))
#
# Para um objeto, calculamos a DISPERSÃO de inclinação
# dos seus vizinhos (±50 km altitude) como proxy de delta_i típico.

print('Calculando dispersao de inclinacao na vizinhanca...')

alt_arr  = gp['altitude_km'].values
incl_arr = gp['inclination_deg'].values
N = len(alt_arr)

incl_std_local  = np.zeros(N, dtype=np.float32)
incl_mean_local = np.zeros(N, dtype=np.float32)

for start in range(0, N, chunk):
    end = min(start + chunk, N)
    alt_chunk  = alt_arr[start:end]
    diffs = np.abs(alt_chunk[:, None] - alt_arr[None, :])
    mask  = diffs <= RAIO_ALT  # (chunk, N)

    for j in range(end - start):
        vizinhos_incl = incl_arr[mask[j]]
        if len(vizinhos_incl) > 1:
            incl_std_local[start + j]  = vizinhos_incl.std()
            incl_mean_local[start + j] = vizinhos_incl.mean()
        else:
            incl_std_local[start + j]  = 0.0
            incl_mean_local[start + j] = incl_arr[start + j]

gp['incl_dispersion_local'] = incl_std_local
gp['incl_mean_local']       = incl_mean_local

# Velocidade relativa proxy: v_orbital * sqrt(2 - 2*cos(incl_dispersion_rad))
incl_disp_rad = np.radians(gp['incl_dispersion_local'].values)
gp['vrel_proxy'] = gp['velocity_km_s'] * np.sqrt(np.maximum(2 - 2 * np.cos(incl_disp_rad), 0))

print('Dispersao de inclinacao calculada.')
print(f'  Dispersao media: {gp["incl_dispersion_local"].mean():.1f} graus')
print(f'  Vrel proxy media: {gp["vrel_proxy"].mean():.2f} km/s')

In [ ]:
# ── Passo 6.4: Gradiente de densidade por altitude ──────────────
#
# Como a densidade muda entre faixas adjacentes?
# Alta variação indica uma "borda" de região densa — mais imprevisível.

# Densidade média por faixa de 50 km
dens_por_faixa = gp.groupby('faixa_50km_str')['local_density_km3'].mean().reset_index()
dens_por_faixa.columns = ['faixa_50km_str', 'dens_media_faixa']

gp = gp.merge(dens_por_faixa, on='faixa_50km_str', how='left')

# Gradiente: diferença de densidade entre faixas adjacentes (valor absoluto)
# Ordenamos as faixas e calculamos diff
faixa_dens = gp.groupby('faixa_centro')['local_density_km3'].mean().sort_index()
faixa_grad = faixa_dens.diff().abs().fillna(0)
faixa_grad_map = faixa_grad.to_dict()

gp['alt_density_gradient'] = gp['faixa_centro'].map(faixa_grad_map).fillna(0)

print('Gradiente de densidade calculado.')

## 7. Definição do Target Final: CPS (Conjunction Proxy Score)

O CPS combina **densidade local de debris** e **velocidade relativa proxy**:

$$\text{CPS} = \text{debris\_density\_km3} \times v_{rel\_proxy}$$

### Por que isso é melhor que o IRR?

1. **Sem pesos arbitrários** — a multiplicação é fisicamente motivada (taxa de colisão ∝ densidade × velocidade relativa)
2. **Por objeto, não por faixa** — captura heterogeneidade dentro da mesma altitude
3. **Sem circularidade** — `debris_density` e `vrel_proxy` são independentes entre si e das features de entrada do modelo
4. **Interpretável** — CPS alto = muitos debris próximos com órbitas cruzadas = alto risco de encontro

### Limitações assumidas

- Ignoramos a seção transversal de colisão (assumimos constante)
- Velocidade relativa é uma aproximação baseada em dispersão de inclinação, não em propagação orbital real
- Não consideramos manobras (payloads podem evitar colisões, debris não)
- É uma estimativa relativa, não uma probabilidade absoluta

In [ ]:
# CPS = debris_density * velocidade_relativa_proxy
# NOTA: debris_density_km3 eh normalizado pelo volume da casca (~6.4e10 km3),
# resultando em valores ~1e-8. log1p(1e-8) ~= 1e-8 -> target colapsa para zero.
# SOLUCAO: escalar pelo mediana dos valores positivos antes do log1p.
# Isso preserva a distribuicao relativa e produz um target ML significativo.

gp['CPS'] = gp['debris_density_km3'] * gp['vrel_proxy']

pos_mask = gp['CPS'] > 0
if pos_mask.sum() == 0:
    raise ValueError('ERRO CRITICO: todos os valores de CPS sao zero. Verificar debris_density_km3 e vrel_proxy.')

CPS_POS_MEDIAN = float(gp.loc[pos_mask, 'CPS'].median())
gp['CPS_scaled'] = gp['CPS'] / CPS_POS_MEDIAN  # mediana positiva -> 1.0
gp['CPS_log'] = np.log1p(gp['CPS_scaled'])

# === VALIDACAO CRITICA ===
cps_std = gp['CPS_log'].std()
cps_near_zero_pct = (gp['CPS_log'] < 1e-6).mean()

print(f'[VALIDACAO] Constante de escala (CPS pos. median): {CPS_POS_MEDIAN:.4e}')
print(f'[VALIDACAO] CPS_log  mean={gp["CPS_log"].mean():.4f}  std={cps_std:.4f}')
print(f'[VALIDACAO] CPS_log  min={gp["CPS_log"].min():.4f}   max={gp["CPS_log"].max():.4f}')
print(f'[VALIDACAO] Zeros em CPS_log: {(gp["CPS_log"]==0).sum():,}')

if cps_std < 1e-6:
    raise ValueError(f'ERRO CRITICO: CPS_log.std() = {cps_std:.2e} < 1e-6. Target degenerado!')
if cps_near_zero_pct > 0.95:
    print(f'AVISO: {cps_near_zero_pct:.1%} dos CPS_log sao ~zero. Target pode ser degenerado.')

print()
print('CPS bruto (antes do escalonamento):')
print(gp['CPS'].describe().to_string())
print()
print('CPS_scaled (apos divisao pela mediana positiva):')
print(gp['CPS_scaled'].describe().to_string())
print()
print('Target CPS_log = log1p(CPS_scaled):')
print(gp['CPS_log'].describe().to_string())

zeros = (gp['CPS'] == 0).sum()
print(f'Objetos com CPS=0 (sem debris vizinho em +-50km): {zeros:,} ({zeros/len(gp):.1%})')


## 8. Visualização das Features e do Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# 1. Distribuição de altitude
axes[0].hist(gp['altitude_km'], bins=80, color='steelblue', edgecolor='white', lw=0.3)
axes[0].set_title('Distribuicao de Altitude', fontweight='bold')
axes[0].set_xlabel('Altitude (km)')
axes[0].set_ylabel('N objetos')

# 2. Densidade local por altitude
axes[1].scatter(gp['altitude_km'], gp['local_density_km3'],
                alpha=0.05, s=3, c='steelblue')
axes[1].set_xlabel('Altitude (km)')
axes[1].set_ylabel('Densidade local (obj/km3)')
axes[1].set_title('Densidade Local vs Altitude', fontweight='bold')
axes[1].set_yscale('log')

# 3. CPS_log por altitude
scatter = axes[2].scatter(gp['altitude_km'], gp['CPS_log'],
                          alpha=0.05, s=3, c=gp['is_debris'], cmap='RdBu_r')
axes[2].set_xlabel('Altitude (km)')
axes[2].set_ylabel('CPS (log1p)')
axes[2].set_title('CPS_log vs Altitude\n(azul=payload, vermelho=debris)', fontweight='bold')

# 4. Distribuição do CPS_log
axes[3].hist(gp.loc[gp['CPS_log'] > 0, 'CPS_log'], bins=60,
             color='#e74c3c', edgecolor='white', lw=0.3)
axes[3].set_title('Distribuicao do CPS_log\n(excluindo zeros)', fontweight='bold')
axes[3].set_xlabel('CPS_log')
axes[3].set_ylabel('N objetos')

# 5. CPS_log por tipo de objeto
tipos = ['PAYLOAD', 'DEBRIS', 'ROCKET BODY']
cps_por_tipo = [gp.loc[gp['OBJECT_TYPE'] == t, 'CPS_log'] for t in tipos]
axes[4].boxplot(cps_por_tipo, labels=tipos, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[4].set_title('CPS_log por Tipo de Objeto', fontweight='bold')
axes[4].set_ylabel('CPS_log')

# 6. Velocidade relativa proxy por altitude
axes[5].scatter(gp['altitude_km'], gp['vrel_proxy'],
                alpha=0.05, s=3, color='#e67e22')
axes[5].set_xlabel('Altitude (km)')
axes[5].set_ylabel('vrel proxy (km/s)')
axes[5].set_title('Velocidade Relativa Proxy vs Altitude', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_v2_01_feature_distributions.png', bbox_inches='tight')
plt.show()
print('Salvo: fig_v2_01_feature_distributions.png')

## 9. Matriz de Correlação

Verifica relações entre features e detecta multicolinearidade.

In [ ]:
features_corr = [
    'altitude_km', 'inclination_deg', 'eccentricity', 'velocity_km_s',
    'period_min', 'bstar_abs', 'local_density_km3', 'debris_density_km3',
    'debris_fraction_local', 'incl_dispersion_local', 'vrel_proxy',
    'alt_density_gradient', 'object_type_code', 'CPS_log'
]
features_corr = [f for f in features_corr if f in gp.columns]

corr_df = gp[features_corr].copy()
# substituir inf por nan
corr_df = corr_df.replace([np.inf, -np.inf], np.nan)
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr_matrix, ax=ax, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, center=0,
            linewidths=0.3, cbar_kws={'label': 'Correlacao de Pearson'})
ax.set_title('Matriz de Correlacao — Features + Target (CPS_log)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_v2_02_correlation_matrix.png', bbox_inches='tight')
plt.show()
print('Salvo: fig_v2_02_correlation_matrix.png')

In [ ]:
# Correlações com o target
print('Correlacoes com CPS_log (target):')
print('=' * 45)
corr_target = corr_matrix['CPS_log'].drop('CPS_log').sort_values(key=abs, ascending=False)
for feat, val in corr_target.items():
    sinal = '+' if val > 0 else '-'
    barra = '#' * int(abs(val) * 30)
    print(f'  {feat:30s}: {val:+.3f}  {sinal}{barra}')

## 10. Dataset Final para ML

### Features selecionadas (sem leakage)

Removemos features que foram usadas para **construir** o target diretamente:
- `local_density_raw` e `debris_density_raw` são versões brutas — usamos as normalizadas por volume
- `CPS` (não-log) é redundante com `CPS_log`

O target é `CPS_log`.

In [ ]:
FEATURES_ML = [
    # Orbitais base
    'altitude_km',
    'inclination_deg',
    'eccentricity',
    'velocity_km_s',
    'period_min',

    # Drag / estabilidade
    'bstar_abs',

    # Densidade local (feature mais informativa)
    'local_density_km3',
    'debris_density_km3',
    'debris_fraction_local',

    # Velocidade relativa e dispersão
    'incl_dispersion_local',
    'vrel_proxy',

    # Gradiente de densidade
    'alt_density_gradient',

    # Tipo de objeto
    'object_type_code',
    'is_debris',
    'is_uncontrolled',
]

TARGET = 'CPS_log'

# Montar dataset final
df_ml = gp[FEATURES_ML + [TARGET, 'NORAD_CAT_ID', 'OBJECT_TYPE', 'faixa_50km_str']].copy()

# Tratar infinitos e NaN restantes
df_ml = df_ml.replace([np.inf, -np.inf], np.nan)

# Verificar NaN
nan_counts = df_ml[FEATURES_ML].isnull().sum()
if nan_counts.sum() > 0:
    print('NaN encontrados:')
    print(nan_counts[nan_counts > 0])
    # Preencher com mediana para features numéricas
    for col in FEATURES_ML:
        if df_ml[col].isnull().any():
            df_ml[col] = df_ml[col].fillna(df_ml[col].median())

print(f'Dataset final: {df_ml.shape[0]:,} objetos x {len(FEATURES_ML)} features')
print(f'Target: {TARGET}')
print(f'\nDistribuicao do target:')
print(df_ml[TARGET].describe().round(4))

In [ ]:
# Salvar como parquet (mais eficiente que CSV para dados numéricos)
try:
    df_ml.to_parquet(PROC_DIR / 'processed_features.parquet', index=False)
    print('Salvo como: data/processed/processed_features.parquet')
except Exception as e:
    # fallback para CSV se pyarrow não estiver instalado
    df_ml.to_csv(PROC_DIR / 'processed_features.csv', index=False)
    print(f'pyarrow nao disponivel ({e})')
    print('Salvo como CSV: data/processed/processed_features.csv')

# Salvar também CSV para inspecção manual
df_ml.to_csv(PROC_DIR / 'processed_features.csv', index=False)
print('Salvo tambem como CSV para inspecao.')
print(f'Shape: {df_ml.shape}')

## 11. Dataset Secundário: Nível Faixa (50 km)

Para comparação com a v1 e para o dashboard.

In [ ]:
df_faixa = gp.groupby('faixa_50km_str').agg(
    alt_centro        = ('altitude_km',          'mean'),
    n_total           = ('altitude_km',          'count'),
    n_debris          = ('is_debris',            'sum'),
    n_uncontrolled    = ('is_uncontrolled',       'sum'),
    local_density_mean= ('local_density_km3',    'mean'),
    debris_density_mean=('debris_density_km3',   'mean'),
    debris_frac_mean  = ('debris_fraction_local','mean'),
    incl_mean         = ('inclination_deg',       'mean'),
    incl_dispersion   = ('incl_dispersion_local', 'mean'),
    vrel_proxy_mean   = ('vrel_proxy',            'mean'),
    CPS_log_mean      = ('CPS_log',               'mean'),
    CPS_log_max       = ('CPS_log',               'max'),
).reset_index()

df_faixa['alt_min'] = df_faixa['faixa_50km_str'].apply(
    lambda x: int(str(x).split('-')[0]) if '-' in str(x) else 0
)
df_faixa = df_faixa.sort_values('alt_min').reset_index(drop=True)

df_faixa.to_csv(PROC_DIR / 'features_por_faixa_50km.csv', index=False)
print(f'Dataset por faixa salvo: {df_faixa.shape}')
print(df_faixa[['faixa_50km_str','n_total','n_debris','local_density_mean','CPS_log_mean']].to_string(index=False))

## 12. Documentação das Escolhas

### Por que CPS é melhor que IRR?

O IRR da v1 era uma soma ponderada com pesos definidos manualmente (50/30/20%).
Isso tem dois problemas sérios:

**1. Circularidade potencial:** se usarmos `densidade` como feature no modelo e o target for `0.5 × densidade_normalizada + ...`, o modelo aprende que `densidade` prediz o target — não porque há uma relação real, mas porque o target *é* a densidade. O R² fica alto por razões erradas.

**2. Não há como validar os pesos:** Por que 50% densidade e não 60%? Não há dado que justifique.

O CPS resolve isso porque:
- A fórmula `densidade_debris × v_rel` vem da física (equação de Kessler)
- Os "pesos" são implícitos na física, não escolhidos por nós
- O target é calculado separadamente das features do modelo (sem leakage)

### Assunções feitas

1. **Seção transversal uniforme:** tratamos todos os objetos como tendo o mesmo tamanho. Na realidade varia enormemente.
2. **Vizinhança 1D:** usamos apenas altitude para definir vizinhança. Uma abordagem 3D consideraria também RAAN e argumento de perigeu.
3. **Velocidade relativa via dispersão de inclinação:** é uma aproximação. A velocidade relativa real requer propagação orbital.
4. **Sem propagação temporal:** os elementos TLE são válidos para a data de download, não para instâncias futuras.

### Limitações

- Com ±50 km como raio, objetos em altitudes bem isoladas têm CPS = 0 artificialmente
- Payloads das megaconstelações (Starlink) inflam a densidade mas têm `is_debris=0`, então o CPS deles pode ser baixo mesmo com muitos vizinhos — isso é correto (eles podem manobrar), mas o *risco de terceiros* causado por eles não é capturado
- O CPS é uma métrica relativa, não uma probabilidade absoluta de colisão

In [ ]:
print('=' * 65)
print('  RESUMO — FEATURE ENGINEERING v2')
print('=' * 65)
print(f'\nObjetos processados    : {len(gp):,}')
print(f'Features criadas       : {len(FEATURES_ML)}')
print(f'Target                 : CPS_log (Conjunction Proxy Score)')
print(f'Raio de vizinhanca     : +-{RAIO_ALT:.0f} km de altitude')
print()
print('Features no dataset final:')
for f in FEATURES_ML:
    print(f'  - {f}')
print()
print('Arquivos gerados:')
print('  data/processed/processed_features.parquet  (dataset ML)')
print('  data/processed/processed_features.csv      (inspecao)')
print('  data/processed/features_por_faixa_50km.csv (nivel faixa)')
print('  outputs/fig_v2_01_feature_distributions.png')
print('  outputs/fig_v2_02_correlation_matrix.png')
print()
print('Proximo passo: 03_modelo_ml_v2.ipynb')
print('  Treinar modelo com X=FEATURES_ML, y=CPS_log')
print('=' * 65)